In [98]:
import pandas as pd
import numpy as np
import os

In [99]:
before_df = pd.read_csv("../data/before_train_v1.csv", index_col=0)
train_df = pd.read_csv("../data/train_v1.csv", index_col=0)
test_df = pd.read_csv("../data/test_v1.csv", index_col=0)

In [100]:
tot_train_df = pd.concat([before_df, train_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)

In [101]:
tot_train_df = pd.concat([tot_train_df, test_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)

In [114]:

TRAIN_START_DATE = '2000-01-10'
TRAIN_END_DATE = '2009-12-31'

TRADE_START_DATE = '2013-01-01'

# TRAIN_START_DATE = '2000-01-10'
# TRAIN_END_DATE = '2023-12-31'
# TRADE_START_DATE = '2010-01-01'
# TRADE_END_DATE = '2023-12-31'

window_days = 252
start_day = 252
top_k = 5
top_pct = 0.5
risk_free_rate = 0
gamma = 10

In [103]:
df_price = tot_train_df[["date", "ticker", "close"]].dropna()


In [104]:
price_df = df_price.pivot(index="date", columns="ticker", values="close")

In [105]:
price_df = price_df.sort_index()

In [106]:
returns = price_df.pct_change().fillna(0)

In [113]:
(1 + returns).cumprod().iloc[-1]

ticker
Australia                   2.543600
Austria                     2.169819
Belgium                     0.851604
Canada                      3.058681
Energy                      3.464946
France                      1.708976
Germany                     1.274850
Gold                        1.601468
Hong Kong                   1.909111
Italy                       1.161981
Japan                       1.571694
Malaysia                    1.369974
Metals_Mining               3.020521
Mexico                      6.687697
Netherlnd                   1.669634
Oil_Gas_Consumable_Fuels    3.830314
SP500                       3.883910
Singapore                   1.519086
Spain                       0.949487
Sweden                      1.981535
Switzrlnd                   2.737585
Utd_Kgdm                    0.815209
Name: 2023-12-29, dtype: float64

In [10]:
import cvxpy as cp

In [11]:
# ✅ 평균-분산 (Mean-Variance), 최소분산 (MinVar), 리스크패리티 (Risk-Parity), Max Sharpe
# 전략들의 차이는 "최적화 기준"의 차이뿐이며, 백테스트 로직 구조는 거의 동일함

import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

# ✅ 평균-분산 최적화 함수 (공매도 금지)
def compute_mv_weights(mu, cov_matrix, gamma=3, max_vol=None):
    n = len(mu)
    x = cp.Variable(n)
    objective = cp.Maximize(mu @ x - (gamma / 2) * cp.quad_form(x, cov_matrix))
    constraints = [cp.sum(x) == 1, x >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve()

    # 👉 Step 2: 원래 문제 다시 정의
    if max_vol is not None:
        constraints.append(cp.quad_form(x, cov_matrix) <= max_vol**2)

    prob = cp.Problem(objective, constraints)
    prob.solve()
    
    return x.value

# ✅ 최소분산 포트폴리오 (Min-Var)
def compute_minvar_weights(mu, cov_matrix):
    n = cov_matrix.shape[0]
    w = cp.Variable(n)
    objective = cp.Minimize(cp.quad_form(w, cov_matrix))
    constraints = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return w.value


# ✅ 리스크 패리티 가중치 (역분산 기반 근사)
def compute_risk_parity_weight_from_window(returns_window):
    var = returns_window.var()
    inv_var = 1 / var
    weights = inv_var / inv_var.sum()
    return weights


def compute_max_sharpe_min_var(mu, cov_matrix, risk_free_rate=0.0, max_vol=None, target_return=0):
    n = len(mu)
    x = cp.Variable(n)
    excess_mu = mu - risk_free_rate
    # objective = cp.Minimize(cp.quad_form(x, cov_matrix))
    w_tilde = cp.Variable(n)  # 치환된 weight (w_tilde = k * w)
    k = cp.Variable(nonneg=True)  # 스케일 변수
    objective = cp.Minimize(cp.quad_form(w_tilde, cov_matrix))

    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x >= target_return]
    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x == 1]
    constraints = [
        excess_mu.T @ w_tilde == 1,   # 초과수익률 고정
        cp.sum(w_tilde) == k,          # w_tilde = k * w
        w_tilde >= 0          # w_tilde = k * w
    ]


    prob = cp.Problem(objective, constraints)
    prob.solve()
    
    # try:
    #     prob.solve()
    #     if x.value is not None:
    #         return x.value
    #     else:
    #         raise ValueError("No solution from solver")
    # except Exception as e:
    #     return np.ones(n) / n  # fallback: equal weights
    if w_tilde.value is not None and k.value is not None and k.value > 0:
        w = w_tilde.value / k.value
        return w
    else:
        print("최적화 실패. 균등 포트폴리오로 fallback.")
        return np.zeros(n)

# ✅ 백테스트 공통 함수 (최적화 함수 인자로 받음)
def backtest_strategy(returns, compute_weights_fn, rebalance_every=20, window_days=252, cost=0.003, start_day=252, **kwargs):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        window_data = returns.iloc[i - window_days:i]
        # print(f"Window data shape: {window_days}")
        if window_data.isna().sum().sum() > 0:
            i += rebalance_every
            continue

        mu = window_data.mean().values
        cov = window_data.cov().values

        # mu = window_data.mean().values
        # cov = window_data.cov().values
        try:
            weights = compute_weights_fn(mu, cov, **kwargs)
        except TypeError:
            weights = compute_weights_fn(window_data, **kwargs)

        if weights is None:
            i += rebalance_every
            continue

        n_assets = returns.shape[1]
        
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
            
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    strat_returns = pd.Series(dict(portfolio_returns)).sort_index()
    strat_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return strat_returns, strat_weights


## 다른 전략

In [12]:
# 3. 리스크 패리티 (Risk-Parity)
rp_returns, rp_weights = backtest_strategy(
    returns,
    compute_weights_fn=compute_risk_parity_weight_from_window,
    window_days=window_days
)

# 2. 최소분산 (Min-Variance)
minvar_returns, minvar_weights = backtest_strategy(
    returns,
    compute_weights_fn=compute_minvar_weights,
    window_days=window_days

)

ms_returns, ms_weights = backtest_strategy(
    returns,
    compute_weights_fn=compute_max_sharpe_min_var,
    risk_free_rate=risk_free_rate,  # 무위험 수익률
    target_return=0.0,
    window_days=window_days

    # max_vol=0.01   # 최대 변동성 제약 (선형 근사)
)


/opt/conda/lib/python3.11/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def backtest_daa_from_pivot(
    returns: pd.DataFrame,
    pivot_score: pd.DataFrame,
    window_days: int = 252,
    rebalance_every: int = 20,
    top_n: int = 10,
    score_threshold: float = 0.0,
    cost: float = 0.003,
    start_day : int = 252
):
    """
    DAA 전략 20일 리밸런싱 백테스트 함수 (거래비용 포함, 모멘텀 스코어 피벗 사용)

    Parameters:
    - returns: 일간 수익률 DataFrame (index: date, columns: tickers)
    - pivot_score: 모멘텀 스코어 DataFrame (index: date, columns: tickers)
    - window_days: 기록용 (사용 X)
    - rebalance_every: 리밸런싱 주기
    - top_n: 상위 n개 종목 선택
    - score_threshold: 모멘텀 스코어 최소 기준
    - cost: 거래비용

    Returns:
    - daa_returns: 전략 수익률 Series
    - daa_weights: 포트폴리오 비중 DataFrame
    """
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = pd.Series(0, index=returns.columns, dtype=float)

    i = start_day
    while i < len(dates) - rebalance_every:
        current_date = dates[i]
        score_today = pivot_score.loc[current_date]

        # 스코어가 0보다 큰 종목 필터링
        satisfied_assets = score_today[score_today > score_threshold]
        if satisfied_assets.empty:
            selected = []
        else:
            selected = satisfied_assets.sort_values(ascending=False).head(top_n).index

        weights = pd.Series(0, index=returns.columns, dtype=float)
        if len(selected) > 0:
            weights[selected] = 1 / len(selected)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
        
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            tc = np.abs(weights - prev_weights).sum() * cost if j == 0 else 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights.copy()))

        prev_weights = weights.copy()
        i += rebalance_every

    daa_returns = pd.Series(dict(portfolio_returns)).sort_index()
    daa_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return daa_returns, daa_weights


In [14]:
# 피벗 테이블 생성
pivot_close = tot_train_df.pivot(index='date', columns='ticker', values='close')
pivot_score = tot_train_df.pivot(index='date', columns='ticker', values='mom_score')
returns = pivot_close.pct_change().fillna(0)

# 백테스트 실행
daa_returns, daa_weights = backtest_daa_from_pivot(returns, pivot_score, top_n=top_k, window_days=window_days, score_threshold=0)
# 성과 시각화
# daa_sharpe = daa_returns.mean() / daa_returns.std() * np.sqrt(252)

In [15]:
def backtest_paa_from_pivot(returns: pd.DataFrame,
                             pivot_score: pd.DataFrame,
                             window_days: int = 252,
                             rebalance_every: int = 20,
                             top_n: int = 10,
                             score_threshold: float = 0.0,
                             cost: float = 0.003,
                             start_day: int = 252):
    """
    PAA 전략 백테스트 (수익률 기반 점수 사용, 거래비용 반영)

    Parameters:
    - returns: 일간 수익률 DataFrame (index: date, columns: tickers)
    - pivot_score: PAA score DataFrame (index: date, columns: tickers)
    - window_days: 과거 데이터 시작 시점
    - rebalance_every: 리밸런싱 주기
    - top_n: 상위 자산 수
    - score_threshold: 점수 필터링 기준
    - cost: 거래 비용 비율

    Returns:
    - paa_returns: 전략 일간 수익률 Series
    - paa_weights: 전략 리밸런싱 시점별 자산 비중 DataFrame
    """
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = pd.Series(0, index=returns.columns, dtype=float)

    i = start_day
    while i < len(dates) - rebalance_every:
        current_date = dates[i]
        score_today = pivot_score.loc[current_date]
        satisfied_assets = score_today[score_today > score_threshold]
        if satisfied_assets.empty:
            selected = []
        else:
            selected = satisfied_assets.sort_values(ascending=False).head(top_n).index

        weights = pd.Series(0, index=returns.columns, dtype=float)
        if len(selected) > 0:
            weights[selected] = 1 / len(selected)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
                    
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            tc = np.abs(weights - prev_weights).sum() * cost if j == 0 else 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights.copy()))

        prev_weights = weights.copy()
        i += rebalance_every

    paa_returns = pd.Series(dict(portfolio_returns)).sort_index()
    paa_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return paa_returns, paa_weights

In [16]:
pivot_close = tot_train_df.pivot(index='date', columns='ticker', values='close')
pivot_paa_score = tot_train_df.pivot(index='date', columns='ticker', values='paa_score')
returns = pivot_close.pct_change().fillna(0)

# 백테스트 실행
paa_returns, paa_weights = backtest_paa_from_pivot(
    returns, pivot_paa_score,
    top_n=top_k,
    window_days = window_days,
    score_threshold=0.0,
    cost=0.003
)


In [17]:
def backtest_gtaa_from_pivot(returns: pd.DataFrame,
                              pivot_close: pd.DataFrame,
                              pivot_sma: pd.DataFrame,
                              window_days: int = 252,
                              rebalance_every: int = 20,
                              top_n: int = 10,
                              cost: float = 0.003,
                              start_day: int = 252):
    """
    GTAA 전략 백테스트 (SMA 기준 + SMA 대비 상대 수익률 정렬, 거래비용 반영)

    Parameters:
    - returns: 일간 수익률 DataFrame (index: date, columns: tickers)
    - pivot_close: 종가 DataFrame (index: date, columns: tickers)
    - pivot_sma: SMA_220 DataFrame (index: date, columns: tickers)
    - window_days: 과거 데이터 시작 시점 (기록용)
    - rebalance_every: 리밸런싱 주기
    - top_n: 상위 자산 수
    - cost: 거래 비용 비율

    Returns:
    - gtaa_returns: 전략 일간 수익률 Series
    - gtaa_weights: 전략 리밸런싱 시점별 자산 비중 DataFrame
    """
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = pd.Series(0, index=returns.columns, dtype=float)

    i = start_day
    while i < len(dates) - rebalance_every:
        current_date = dates[i]
        close_today = pivot_close.loc[current_date]
        sma_today = pivot_sma.loc[current_date]

        # 조건: 현재 종가가 SMA보다 큰 자산
        condition = (close_today > sma_today)
        filtered_assets = close_today[condition]

        # 정렬 기준: SMA 대비 수익률 비율 (close / sma - 1)
        relative_returns = (filtered_assets / sma_today[condition]) - 1
        selected = relative_returns.sort_values(ascending=False).head(top_n).index if not relative_returns.empty else []

        # 포트폴리오 비중 계산
        weights = pd.Series(0, index=returns.columns, dtype=float)
        if len(selected) > 0:
            weights[selected] = 1 / len(selected)
        else:
            selected = []

        # 수익률 계산
        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
                    
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            tc = np.abs(weights - prev_weights).sum() * cost if j == 0 else 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights.copy()))

        prev_weights = weights.copy()
        i += rebalance_every

    gtaa_returns = pd.Series(dict(portfolio_returns)).sort_index()
    gtaa_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    gtaa_weights.index.name = 'date'
    return gtaa_returns, gtaa_weights


In [18]:
pivot_close = tot_train_df.pivot(index='date', columns='ticker', values='close')
pivot_sma = tot_train_df.pivot(index='date', columns='ticker', values='SMA_220')
returns = pivot_close.pct_change().fillna(0)


gtaa_returns, gtaa_weights = backtest_gtaa_from_pivot(
    returns,
    pivot_close,
    pivot_sma,
    window_days=window_days,
    top_n=top_k,
    cost=0.003
)

In [19]:
def backtest_equal_weight_20day(returns: pd.DataFrame, rebalance_every=20, window_days=252, cost=0.003, start_day=252):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        rebalance_day = dates[i]
        window_data = returns.iloc[i - window_days:i]


        n_assets = window_data.shape[1]
        weights = np.ones(n_assets) / n_assets  # 1/N 포트폴리오
        n_assets = returns.shape[1]
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)  # 최초 리밸런싱: 전량 매수로 간주
            
        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]

        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    ew_returns = pd.Series(dict(portfolio_returns)).sort_index()
    ew_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return ew_returns, ew_weights


In [20]:
# 백테스트 실행
equal_returns, equal_weights = backtest_equal_weight_20day(returns, rebalance_every=20, window_days=window_days, cost=0.003)

# Sharpe 비율 계산
# equal_sharpe = equal_returns.mean() / equal_returns.std() * np.sqrt(252)

In [21]:
strategy_returns = {
    'DAA': daa_returns,
    'PAA': paa_returns,
    'GTAA': gtaa_returns,
    'Risk Parity': rp_returns,
    'min-Variance': minvar_returns,
    "Mean-Variance_max_sharpe": ms_returns,
    'Equal Weight': equal_returns
}

In [23]:
daa_cum = (1 + daa_returns).cumprod()
paa_cum = (1 + paa_returns).cumprod()
gtaa_cum = (1 + gtaa_returns).cumprod()
rp_cum = (1 + rp_returns).cumprod()
minvar_cum = (1 + minvar_returns).cumprod()
ms_cum = (1 + ms_returns).cumprod()



In [24]:
import os
import sys
# 현재 경로의 부모 디렉토리
parent_dir = os.path.dirname(os.getcwd())

# sys.path에 현재 경로와 부모 경로 추가
sys.path.append(parent_dir)
import eval_metric as em

In [25]:
risk_free_rate = 0.0
annual_factor = 252

In [26]:
def calculate_annual_return(returns, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    cumulative = np.prod(1 + returns)
    n_periods = len(returns)
    return cumulative ** (annual_factor / n_periods) - 1

def calculate_max_drawdown(returns):
    returns = np.array(returns, dtype=np.float64)
    cumulative_returns = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdowns = (cumulative_returns - running_max) / running_max
    return abs(np.min(drawdowns))

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)


def calculate_sharpe_ratio(returns, risk_free_rate=0.02, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    annual_return = calculate_annual_return(returns, annual_factor)
    annual_std_dev = calculate_volatility(returns, annual_factor)
    excess_return =  (annual_return - risk_free_rate) 
    # 연환산 수익률과 표준편차
    return excess_return / annual_std_dev if annual_std_dev != 0 else 0.0


def calculate_annualized_sortino_ratio(returns, risk_free_rate=0.02, annual_factor=252):
    """
    Annualized Sortino Ratio = (Mean Portfolio Return - Risk-Free Rate) * Annual Factor / Downside Deviation
    """
    annual_return = calculate_annual_return(returns, annual_factor)
    excess_return =  (annual_return - risk_free_rate) 

    downside_returns = returns[returns < 0]
    downside_std = np.std(downside_returns, ddof=1) if len(downside_returns) > 1 else 0.0

    return excess_return / (downside_std * np.sqrt(annual_factor)) if downside_std != 0 else 0.0


def calculate_calmar_ratio(returns, annual_factor=252):
    """
    Calmar Ratio = Annualized Return / Maximum Drawdown
    """
    returns = np.array(returns, dtype=np.float64)
    annual_return = calculate_annual_return(returns, annual_factor)
    max_drawdown = calculate_max_drawdown(returns)
    
    return  (annual_return) / max_drawdown if max_drawdown != 0 else 0.0  # MDD가 0이면 0 반환


In [27]:
def calculate_rolling_metrics(
    returns: pd.Series,
    windows: list = [20, 252],
    risk_free_rate: float = 0.02,
    annual_factor: int = 252,
    name : str = "returns",
    shift_features: bool = True,  # ✅ 추가

) -> pd.DataFrame:
    """
    주어진 daily return 시리즈에 대해 rolling window 기반 
    Sharpe, Volatility, Sortino, Calmar 계산

    Args:
        returns (pd.Series): 일간 수익률
        windows (list): rolling window list (e.g., [20, 252])
        risk_free_rate (float): 무위험 수익률 (연환산)
        annual_factor (int): 연환산 계수 (default 252)

    Returns:
        pd.DataFrame: 원본 daily_return + 모든 rolling metric columns
    """
    result = pd.DataFrame({'daily_return': returns})
    result["tic"] = name

    for window in windows:
        result[f'sharpe_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_sharpe_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'vol_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_volatility(x, annual_factor),
            raw=False
        )
        result[f'sortino_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_annualized_sortino_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'calmar_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_calmar_ratio(x, annual_factor),
            raw=False
        )
    if shift_features:
        feature_cols = [col for col in result.columns if col not in ['daily_return', 'tic']]
        result[feature_cols] = result[feature_cols].shift(1)
        
    return result

In [28]:
daa_eval = calculate_rolling_metrics(daa_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "DAA")
paa_eval = calculate_rolling_metrics(paa_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "PAA")
gtaa_eval = calculate_rolling_metrics(gtaa_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "GTAA")
rp_eval = calculate_rolling_metrics(rp_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "Risk Parity")
minvar_eval = calculate_rolling_metrics(minvar_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "min-Variance")
ms_eval = calculate_rolling_metrics(ms_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "Mean-Variance_max_sharpe")
ew_eval = calculate_rolling_metrics(equal_returns, windows=[20, 252], risk_free_rate = risk_free_rate, annual_factor = annual_factor, name = "Equal Weight")

In [29]:
portfolio_price = pd.DataFrame({
    'DAA': daa_cum,
    'PAA': paa_cum,
    'GTAA': gtaa_cum,
    'Risk Parity': rp_cum,
    'min-Variance': minvar_cum,
    "Mean-Variance_max_sharpe": ms_cum
})

In [30]:
daa_df = pd.DataFrame({
    'close': daa_cum,
    "tic" : "DAA"
})
paa_df = pd.DataFrame({
    'close': paa_cum,
    "tic" : "PAA"
})
gtaa_df = pd.DataFrame({
    'close': gtaa_cum,
    "tic" : "GTAA"
})
rp_df = pd.DataFrame({
    'close': rp_cum,
    "tic" : "Risk Parity"
})
minvar_df = pd.DataFrame({
    'close': minvar_cum,
    "tic" : "min-Variance"
})
ms_df = pd.DataFrame({
    'close': ms_cum,
    "tic" : "Mean-Variance_max_sharpe"
})

In [31]:
import pandas as pd

def calculate_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    주어진 데이터프레임에 대해 모멘텀 관련 특성 생성.
    
    Args:
        df (pd.DataFrame): 종가(close) 컬럼을 포함한 데이터프레임
        close_col (str): 종가 컬럼 이름 (default: 'close')
        
    Returns:
        pd.DataFrame: 모멘텀 특성이 추가된 데이터프레임
    """
    data = df.copy()
    
    # 설정할 period 리스트
    periods = [20, 40, 60, 80, 100, 120, 140, 160, 180, 220, 240, 252]
    columns_list = ["return_1m", "return_3m", "return_6m", "return_12m", "return_avg", "mom_12m", "mom_score"]

    # 기간별 이전 종가 저장
    for idx, period in enumerate(periods):
        col_name = f'close_{idx+1}_month'
        data[col_name] = data['close'].shift(period)
        columns_list.append(col_name)
    
    # 모멘텀 및 수익률 계산
    data['return_1m'] = (data['close'] - data['close_1_month']) / data['close_1_month']
    data['return_3m'] = (data['close'] - data['close_3_month']) / data['close_3_month']
    data['return_6m'] = (data['close'] - data['close_6_month']) / data['close_6_month']
    data['return_12m'] = (data['close'] - data['close_12_month']) / data['close_12_month']
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    data["mom_score"] = (
        12 * data["return_1m"] +
        4 * data["return_3m"] +
        2 * data["return_6m"] +
        1 * data["return_12m"]
    ) / 19

    # 평균 return
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    
    # 모멘텀 점수 (mom_score)
    data['mom_score'] = (
        12 * data['return_1m'] +
        4 * data['return_3m'] +
        2 * data['return_6m'] +
        1 * data['return_12m']
    ) / 19
    
    return data


In [32]:
daa_df = calculate_momentum_features(daa_df)
paa_df = calculate_momentum_features(paa_df)
gtaa_df = calculate_momentum_features(gtaa_df)
rp_df = calculate_momentum_features(rp_df)
min_var_df = calculate_momentum_features(minvar_df)
ms_df = calculate_momentum_features(ms_df)

In [33]:
daa_df["ret"] = daa_df["close"].pct_change()
paa_df["ret"] = paa_df["close"].pct_change()
gtaa_df["ret"] = gtaa_df["close"].pct_change()
rp_df["ret"] = rp_df["close"].pct_change()
min_var_df["ret"] = minvar_df["close"].pct_change()
ms_df["ret"] = ms_df["close"].pct_change()

In [34]:
eval_feature = ['vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [35]:
daa_df = pd.concat([daa_df, daa_eval[eval_feature]], axis=1)
paa_df = pd.concat([paa_df, paa_eval[eval_feature]], axis=1)
gtaa_df = pd.concat([gtaa_df, gtaa_eval[eval_feature]], axis=1)
rp_df = pd.concat([rp_df, rp_eval[eval_feature]], axis=1)
min_var_df = pd.concat([min_var_df, minvar_eval[eval_feature]], axis=1)
ms_df = pd.concat([ms_df, ms_eval[eval_feature]], axis=1)

In [36]:
portfolio_df = pd.concat([daa_df, paa_df, gtaa_df, rp_df, min_var_df, ms_df], axis=0)

In [37]:
portfolio_df.reset_index(inplace=True)

In [38]:
portfolio_df.rename(columns={"index": "date"}, inplace=True)

In [39]:
portfolio_df.tic.unique()

array(['DAA', 'PAA', 'GTAA', 'Risk Parity', 'min-Variance',
       'Mean-Variance_max_sharpe'], dtype=object)

In [40]:
imp = portfolio_df[portfolio_df.tic == "PAA"].copy()


In [41]:
portfolio_df.columns

Index(['date', 'close', 'tic', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252'],
      dtype='object')

In [42]:
select_features = ['return_1m',
       'return_3m', 'return_6m', 'return_12m', 'return_avg',
       'ret', 'vol_20', 'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [43]:
imp = portfolio_df[portfolio_df.tic == "DAA"].copy()
imp.reset_index(drop=True, inplace=True)

In [44]:
imp["ret"].argmax()

np.int64(5091)

In [45]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp[select_features].describe())

DAA
         return_1m    return_3m    return_6m   return_12m   return_avg  \
count  6009.000000  5969.000000  5909.000000  5777.000000  6009.000000   
mean      0.006303     0.018367     0.038532     0.085939     0.035947   
std       0.054464     0.089424     0.126493     0.186373     0.092475   
min      -0.279375    -0.267377    -0.334508    -0.365015    -0.259438   
25%      -0.023288    -0.037057    -0.050201    -0.050363    -0.029860   
50%       0.004141     0.013171     0.020379     0.066434     0.033953   
75%       0.035108     0.073929     0.129855     0.202583     0.093845   
max       0.407338     0.551871     0.693333     1.144009     0.444354   

               ret       vol_20   sharpe_252      vol_252  sortino_252  \
count  6028.000000  6009.000000  5777.000000  5777.000000  5777.000000   
mean      0.000315     0.170110     0.489865     0.184850     0.739184   
std       0.012248     0.094734     1.012900     0.062890     1.490346   
min      -0.112263     0.000000  

In [46]:
portfolio_df.sort_values(by=["date", "tic"], inplace=True)

In [47]:
portfolio_df.reset_index(drop=True, inplace=True)

In [48]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.describe())

DAA
             close  close_1_month  close_2_month  close_3_month  \
count  6029.000000    6009.000000    5989.000000    5969.000000   
mean      2.290994       2.284496       2.278406       2.271328   
std       0.992192       0.987404       0.983393       0.977358   
min       0.802058       0.802058       0.802058       0.802058   
25%       1.575836       1.571168       1.566441       1.561790   
50%       2.401749       2.400161       2.399057       2.396768   
75%       2.638878       2.636325       2.634114       2.629681   
max       5.111728       5.111728       5.111728       5.111728   

       close_4_month  close_5_month  close_6_month  close_7_month  \
count    5949.000000    5929.000000    5909.000000    5889.000000   
mean        2.263475       2.255543       2.247276       2.239306   
std         0.969540       0.961486       0.952532       0.944256   
min         0.802058       0.802058       0.802058       0.802058   
25%         1.551498       1.519161       1.497

In [49]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.isna().sum())

DAA
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m          60
return_6m         120
return_12m        252
return_avg         20
mom_score         252
ret                 1
vol_20             20
sharpe_252        252
vol_252           252
sortino_252       252
calmar_252        252
dtype: int64
GTAA
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m          60
return_6m 

In [50]:
portfolio_na = portfolio_df.dropna()

In [51]:
portfolio_na.reset_index(drop=True, inplace=True)

In [52]:
for tic  in portfolio_na.tic.unique():
    imp = portfolio_na[portfolio_na.tic == tic].copy()
    print(tic)
    print(imp.date.min())
    print(imp.date.max())

DAA
2001-01-10
2023-12-29
GTAA
2001-01-10
2023-12-29
Mean-Variance_max_sharpe
2001-01-10
2023-12-29
PAA
2001-01-10
2023-12-29
Risk Parity
2001-01-10
2023-12-29
min-Variance
2001-01-10
2023-12-29


In [53]:
portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)

/tmp/ipykernel_1817813/1574858568.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)


In [54]:
TRAIN_START_DATE = '2001-01-10'
TRAIN_END_DATE = '2010-12-31'
VAILD_START_DATE = '2011-01-01'
VAILD_END_DATE = '2016-12-31'
TEST_START_DATE = '2017-01-01'
TEST_END_DATE = '2023-12-31'

In [55]:
def data_split(df, start, end, target_date_col="date"):
    """
    split the dataset into training or testing using date
    :param data: (df) pandas dataframe, start, end
    :return: (df) pandas dataframe
    """
    data = df[(df[target_date_col] >= start) & (df[target_date_col] <= end)]
    data = data.sort_values([target_date_col, "ticker"], ignore_index=True)
    # data.index = data[target_date_col].factorize()[0]
    return data

In [56]:
train = data_split(portfolio_na, TRAIN_START_DATE,TRAIN_END_DATE)
vaild = data_split(portfolio_na, VAILD_START_DATE,VAILD_END_DATE)
test = data_split(portfolio_na, TEST_START_DATE,TEST_END_DATE)

In [57]:
def min_max_normalize_by_ticker_train_test(train_df, vaild_df, test_df, columns):
    """
    주어진 컬럼들을 train 기준으로 티커별 min-max 정규화
    
    Parameters:
        train_df (DataFrame): 학습용 데이터
        test_df (DataFrame): 테스트용 데이터
        columns (list): 정규화할 컬럼 리스트
        
    Returns:
        train_df, test_df: 정규화된 결과가 포함된 데이터프레임
    """
    # 티커별로 정규화 통계 계산
    stats = train_df.groupby("ticker")[columns].agg(["min", "max"])
    
    # 컬럼명 정리 (MultiIndex → flat column name)
    stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]

    # train/test에 붙이기
    train_df = train_df.merge(stats, on="ticker", how="left")
    vaild_df = vaild_df.merge(stats, on="ticker", how="left")
    test_df = test_df.merge(stats, on="ticker", how="left")

    # 컬럼별 정규화 수행
    for col in columns:
        min_col = f"{col}_min"
        max_col = f"{col}_max"
        norm_col = f"{col}_norm"

        train_df[norm_col] = (train_df[col] - train_df[min_col]) / (train_df[max_col] - train_df[min_col])
        vaild_df[norm_col] = (vaild_df[col] - vaild_df[min_col]) / (vaild_df[max_col] - vaild_df[min_col])
        test_df[norm_col] = (test_df[col] - test_df[min_col]) / (test_df[max_col] - test_df[min_col])

    # 불필요한 min/max 컬럼 제거
    cols_to_drop = [f"{col}_min" for col in columns] + [f"{col}_max" for col in columns]
    train_df.drop(columns=cols_to_drop, inplace=True)
    vaild_df.drop(columns=cols_to_drop, inplace=True)
    test_df.drop(columns=cols_to_drop, inplace=True)

    return train_df, vaild_df, test_df


In [58]:
columns_to_normalize = ["ret", 'close', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score',  'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [59]:
train_df, vaild_df, test_df = min_max_normalize_by_ticker_train_test(train, vaild, test, columns_to_normalize)

In [60]:
for tic in test_df.ticker.unique():
    imp = test_df[test_df.ticker == tic].copy()
    print(tic)
    print(imp.date.min())
    print(imp.date.max())

DAA
2017-01-03
2023-12-29
GTAA
2017-01-03
2023-12-29
Mean-Variance_max_sharpe
2017-01-03
2023-12-29
PAA
2017-01-03
2023-12-29
Risk Parity
2017-01-03
2023-12-29
min-Variance
2017-01-03
2023-12-29


In [61]:
train_df.to_csv("../data/portfolio_price/train_portfolio_price_v1.csv", index=False)
vaild_df.to_csv("../data/portfolio_price/vaild_portfolio_price_v1.csv", index=False)
test_df.to_csv("../data/portfolio_price/test_portfolio_price_v1.csv", index=False)

In [87]:
train_df.columns

Index(['date', 'close', 'ticker', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252', 'ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm'],
      dtype='object')

In [63]:
select_features = ['ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm']

In [88]:
train_df.to_csv("../data/portfolio_price/train_portfolio_price_v2.csv", index=False)
vaild_df.to_csv("../data/portfolio_price/vaild_portfolio_price_v2.csv", index=False)
test_df.to_csv("../data/portfolio_price/test_portfolio_price_v2.csv", index=False)

In [64]:
tot_data = pd.concat([train_df, vaild_df, test_df])

In [65]:
tot_data = tot_data.sort_values(["date", "ticker"])

In [66]:
describe_df = tot_data[select_features].describe(include='all')

In [67]:
describe_df = vaild_df[select_features].describe(include='all')

In [68]:
# 고유 날짜/종목 추출 및 정렬
dates = np.sort(tot_data['date'].unique())
tics = np.sort(tot_data['ticker'].unique())

In [69]:
# 인덱스 매핑 (빠른 접근용)
date2idx = {d: i for i, d in enumerate(dates)}
tic2idx = {t: i for i, t in enumerate(tics)}

In [70]:
state_array = np.zeros((len(dates), len(tics), len(select_features)), dtype=np.float32)

In [71]:
for row in tot_data.itertuples():
    d_idx = date2idx[row.date]
    t_idx = tic2idx[row.ticker]
    f_vals = [getattr(row, f) for f in select_features]
    state_array[d_idx, t_idx, :] = np.nan_to_num(f_vals)  # NaN은 0으로

In [72]:
tot_data[tot_data["date"] == "2001-01-10"][select_features]

,ret_norm,close_norm,return_1m_norm,return_3m_norm,return_6m_norm,return_12m_norm,return_avg_norm,mom_score_norm,vol_20_norm,sharpe_252_norm,vol_252_norm,sortino_252_norm,calmar_252_norm
0,0.497235,0.051749,0.480091,0.460559,0.220733,0.281655,0.349670,0.482429,0.245513,0.207257,0.344309,0.187265,0.076811
1,0.479326,0.166503,0.522945,0.357121,0.217147,0.150413,0.321693,0.478874,0.412346,0.154794,0.609022,0.169126,0.047078
2,0.480634,0.088750,0.419052,0.349657,0.018104,0.139390,0.125510,0.317749,0.360343,0.106662,0.617473,0.155364,0.044698
3,0.376955,0.125742,0.471157,0.362501,0.190063,0.142059,0.294619,0.471654,0.499061,0.151375,0.759020,0.172882,0.033926
4,0.390383,0.184236,0.576988,0.481288,0.347753,0.315226,0.554019,0.618023,0.141763,0.155724,0.187216,0.117548,0.018939
5,0.523068,0.135240,0.555860,0.466902,0.355193,0.273902,0.493440,0.586364,0.130356,0.113399,0.224607,0.064794,0.007407


In [73]:
TRAIN_START_DATE = '2001-01-10'
TRAIN_END_DATE = '2010-12-31'
VAILD_START_DATE = '2011-01-01'
VAILD_END_DATE = '2016-12-31'
TEST_START_DATE = '2017-01-01'
TEST_END_DATE = '2023-12-31'

In [74]:
valid_start_date = "2011-01-03"
test_start_date = "2017-01-03"

In [75]:
date2idx['2016-12-30']

4016

In [76]:
train = state_array[:2508]

In [77]:
valid = state_array[2508:4017]

In [78]:
test = state_array[4017:]

In [79]:
import torch

In [80]:
train_tensor = torch.from_numpy(train)
valid_tensor = torch.from_numpy(valid)
test_tensor = torch.from_numpy(test)

In [81]:
train.shape

(2508, 6, 13)

In [82]:
train_tensor.shape

torch.Size([2508, 6, 13])

In [83]:
valid_tensor.shape

torch.Size([1509, 6, 13])

In [84]:
torch.save(train_tensor, "../data/portfolio_price/portfolio_train_v1.pt")
torch.save(test_tensor,  "../data/portfolio_price/portfolio_test_v1.pt")
torch.save(valid_tensor, "../data/portfolio_price/portfolio_valid_v1.pt")

In [85]:
def calculate_annual_return(returns, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    cumulative = np.prod(1 + returns)
    n_periods = len(returns)
    return cumulative ** (annual_factor / n_periods) - 1


def calculate_max_drawdown(returns):
    returns = np.array(returns, dtype=np.float64)
    cumulative_returns = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdowns = (cumulative_returns - running_max) / running_max
    return abs(np.min(drawdowns))

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)

def calculate_sharpe_ratio(returns, risk_free_rate=0.02, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    annual_return = calculate_annual_return(returns, annual_factor)
    annual_std_dev = calculate_volatility(returns, annual_factor)
    excess_return =  (annual_return - risk_free_rate) 
    # 연환산 수익률과 표준편차
    return excess_return / annual_std_dev if annual_std_dev != 0 else 0.0


def calculate_cumulative_return(returns):
    returns = np.array(returns, dtype=np.float64)
    return np.prod(1 + returns) - 1

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)


# 성과 지표 계산 함수 (파라미터화)
def calculate_performance_metrics(returns, annual_factor=252, risk_free_rate=0.0):
    cumulative_returns = (1 + returns).cumprod()
    # cagr = calculate_cagr(returns, annual_factor)
    annual_return = calculate_annual_return(returns, annual_factor)
    sharpe_ratio = calculate_sharpe_ratio(returns, risk_free_rate, annual_factor)
    mdd = calculate_max_drawdown(returns)
    volatility = calculate_volatility(returns, annual_factor)

    return {
        'Cumulative Return': cumulative_returns.iloc[-1] - 1,
        'Annual Return': annual_return,
        # 'CAGR': cagr,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'MDD': mdd
    }


# 성과 지표 요약 생성 함수 (파라미터 전달)
def get_performance_summary(returns_dict, annual_factor=252, risk_free_rate=0.0):
    metrics_df = pd.DataFrame()
    for name, returns in returns_dict.items():
        metrics_df[name] = calculate_performance_metrics(returns, annual_factor, risk_free_rate)
    return metrics_df.T


In [86]:
metrics_summary = get_performance_summary(
    strategy_returns,
    annual_factor=252,
    risk_free_rate=0.0
)

display(metrics_summary)


,Cumulative Return,Annual Return,Volatility,Sharpe Ratio,MDD
DAA,3.315670,0.063026,0.194447,0.324128,0.394757
PAA,2.433811,0.052918,0.194504,0.272064,0.437553
GTAA,2.133960,0.048904,0.195614,0.250002,0.470740
Risk Parity,1.057583,0.030618,0.160344,0.190952,0.598891
min-Variance,0.794933,0.024752,0.121083,0.204420,0.490862
Mean-Variance_max_sharpe,1.622761,0.041126,0.194177,0.211796,0.476151
Equal Weight,1.525976,0.039491,0.174748,0.225988,0.595042
